In [ ]:
import re
import csv
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen

# The website we are scraping from
base_url = "https://stackoverflow.com/questions?page={}"

# A user-agent to make our request look like it's coming from a real browser
headers = {'User-Agent': 'Mozilla/5.0'}

# Counters for each year
year_count = {"2024": 0, "2023": 0, "2022": 0}
max_questions = {"2024": 10000, "2023": 5000, "2022": 5000}

# Open a CSV file to store our results
with open("stackoverflow_questions.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Question Title", "Year", "Relative Time", "Language"])

    for page in range(10000, 21000):
        try:
            request = Request(base_url.format(page), headers=headers)
            response = urlopen(request)
            soup = BeautifulSoup(response, "html.parser")
        except:
            continue

        questions = soup.find_all("div", class_="s-post-summary")

        for question in questions:
            title_element = question.find("h3")
            question_title = title_element.text.strip() if title_element else "Unknown"

            tags = question.find_all("a", class_="post-tag")
            tag_list = [tag.text.strip() for tag in tags]

            time_element = question.find("span", class_="relativetime")
            relative_time = time_element["title"] if time_element else "Unknown"

            match = re.search(r"(\d{4})", relative_time)
            year = match.group(1) if match else "Unknown"

            if year in max_questions and year_count[year] < max_questions[year]:
                for tag in tag_list or ["Unknown"]:
                    writer.writerow([question_title, year, relative_time, tag])
                year_count[year] += 1

            if all(year_count[y] >= max_questions[y] for y in max_questions):
                print("✅ Scraping done! Saved to 'stackoverflow_questions.csv'.")
                exit()

print("✅ Scraping done! Saved to 'final_data_stackoverflow_questions.csv'.")